In [1]:
import subprocess
subprocess.run(['pip', 'install', 'chromadb', 'sentence-transformers', 'groq', '--quiet'])
print('Dependencies installed')

Dependencies installed


In [2]:
%pip install groq

from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import chromadb
from groq import Groq
from sentence_transformers import SentenceTransformer

INSTITUTE       = 'msrit'
CHROMA_PATH     = '../outputs/chroma_db'
COLLECTION_NAME = 'curriculum_intelligence'
EMBED_MODEL     = 'all-MiniLM-L6-v2'
GROQ_MODEL      = 'llama-3.1-8b-instant'
TOP_K           = 8
TOTAL_COMPANIES = 10

load_dotenv()
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

groq_client = Groq(api_key=GROQ_API_KEY)

try:
    test = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': 'reply with the single word: ready'}],
        max_tokens=5
    )
    print(f'Groq connected — model: {GROQ_MODEL}')
    print(f'Test response: {test.choices[0].message.content.strip()}')
except Exception as e:
    print(f'Groq connection failed: {e}')
    print('Check your GROQ_API_KEY')

print('Imports OK')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\parni\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq connected — model: llama-3.1-8b-instant
Test response: Ready
Imports OK


In [ ]:
analysis     = pd.read_csv('../outputs/skill_analysis.csv')
syllabus     = pd.read_csv('../outputs/cleaned_syllabus.csv')
cooccurrence = pd.read_csv('../outputs/skill_cooccurrence.csv')

syllabus['Institute'] = syllabus['Institute'].str.lower().str.strip()

documents = []

for _, row in analysis.iterrows():
    taught     = 'currently taught at MSRIT' if row['Is_Taught_MSRIT'] == 1 \
                 else 'NOT taught at MSRIT — curriculum gap'
    watch      = ' On the Early Warning Watch List.' \
                 if row.get('Watch_List', 0) == 1 else ''
    transition = f" Transitioning: {row['Transition']}." \
                 if row.get('In_Transition', 0) == 1 else ''
    cooc       = f" Co-occurs with: {row['Top_Cooccurring']}." \
                 if pd.notna(row.get('Top_Cooccurring', '')) \
                 and str(row.get('Top_Cooccurring', '')) != '' else ''
    prop       = f" Propagated gap score: {row['Propagated_Gap_Score']:.3f}." \
                 if row.get('Propagated_Gap_Score', 0) > 0 else ''
    inst_gap   = f" Missing from {int(row.get('Institutional_Gap_Score', 0) * 5)}/5 institutes."

    text = (
        f"Skill: {row['Skill']}. "
        f"Trajectory: {row['Trajectory']}. "
        f"Status: {taught}. "
        f"Demanded by {int(row['Company_Spread'])} of {TOTAL_COMPANIES} companies, "
        f"total frequency {int(row['Freq'])}. "
        f"Trend slope: {row['Trend_Slope']:.2f}. "
        f"Velocity: {row['Velocity']:.2f}. "
        f"Curriculum Lag Index: {int(row['CLI'])} years. "
        f"Priority Score: {row['Priority_Score']:.3f}. "
        f"Debt Score: {row['Debt_Score']:.3f}."
        f"{inst_gap}{watch}{transition}{cooc}{prop}"
    )

    documents.append({
        'id': f"skill_{row['Skill'].replace(' ', '_').replace('/', '_')}",
        'text': text,
        'metadata': {
            'type':           'skill_card',
            'skill':          str(row['Skill']),
            'trajectory':     str(row['Trajectory']),
            'is_gap':         int(row['Is_Taught_MSRIT'] == 0),
            'priority_score': float(row['Priority_Score']),
            'debt_score':     float(row['Debt_Score']),
            'cli':            int(row['CLI']),
            'watch_list':     int(row.get('Watch_List', 0)),
        }
    })

for traj in ['Traditional', 'Transitional', 'Upcoming']:
    group     = analysis[analysis['Trajectory'] == traj]
    gap_group = group[group['Is_Taught_MSRIT'] == 0]
    top5      = gap_group.sort_values('Priority_Score', ascending=False)['Skill'].head(5).tolist()
    desc = {
        'Traditional':  'Established skills, may have declining industry relevance.',
        'Transitional': 'Stable skills shifting in how they are applied.',
        'Upcoming':     'Emerging rapidly — highest urgency for curriculum addition.',
    }[traj]
    text = (
        f"Trajectory group: {traj}. {desc} "
        f"{len(group)} total skills, {len(gap_group)} not taught at MSRIT. "
        f"Average CLI: {group['CLI'].mean():.1f} years. "
        f"Average Priority Score: {group['Priority_Score'].mean():.3f}. "
        f"Top gap skills by priority: {', '.join(top5)}."
    )
    documents.append({
        'id': f"trajectory_{traj.lower()}",
        'text': text,
        'metadata': {'type': 'trajectory_summary', 'trajectory': traj}
    })

if 'Watch_List' in analysis.columns and 'Watch_Urgency' in analysis.columns:
    watch_df = analysis[
        (analysis['Watch_List'] == 1) &
        (analysis['Is_Taught_MSRIT'] == 0)
    ].sort_values('Watch_Urgency', ascending=False)

    if len(watch_df) > 0:
        entries = ' | '.join([
            f"{r['Skill']} (urgency {r.get('Watch_Urgency',0):.3f}, "
            f"{int(r['Company_Spread'])} companies, {r['Trajectory']})"
            for _, r in watch_df.iterrows()
        ])
        documents.append({
            'id': 'watch_list_msrit',
            'text': (
                f"Early Warning Watch List for MSRIT — {len(watch_df)} emerging skills "
                f"not yet mainstream but rising fast. Faculty should plan additions now. "
                f"Skills ranked by urgency: {entries}."
            ),
            'metadata': {'type': 'watch_list', 'institute': 'msrit'}
        })

for (institute, course), grp in syllabus.groupby(['Institute', 'Course']):
    skills_list = grp['Skill'].tolist()
    semester    = str(grp['Semester'].iloc[0]) if 'Semester' in grp.columns else 'unknown'
    documents.append({
        'id': f"syllabus_{institute}_{course.replace(' ', '_')}",
        'text': (
            f"Institute: {institute.upper()}. "
            f"Course: {course} (Semester {semester}). "
            f"Skills taught: {', '.join(skills_list)}. "
            f"Total: {len(skills_list)} skills."
        ),
        'metadata': {
            'type':      'syllabus_card',
            'institute': str(institute),
            'course':    str(course),
            'semester':  semester,
        }
    })

gap_df       = analysis[analysis['Is_Taught_MSRIT'] == 0]
top_priority = gap_df.sort_values('Priority_Score', ascending=False).head(10)['Skill'].tolist()
top_debt     = gap_df.sort_values('Debt_Score', ascending=False).head(10)['Skill'].tolist()
in_trans     = gap_df[gap_df['In_Transition'] == 1]['Skill'].tolist() \
               if 'In_Transition' in gap_df.columns else []

documents.append({
    'id': 'gap_summary_msrit',
    'text': (
        f"MSRIT curriculum gap summary. "
        f"{len(gap_df)} of 95 industry skills are not taught at MSRIT. "
        f"Data from {TOTAL_COMPANIES} companies, 2021-2025. "
        f"Top 10 by Priority Score: {', '.join(top_priority)}. "
        f"Top 10 by Debt Score: {', '.join(top_debt)}. "
        f"Skills in transition: {', '.join(in_trans) if in_trans else 'none'}. "
        f"Average CLI: {gap_df['CLI'].mean():.1f} years. "
        f"Total curriculum debt: {gap_df['Debt_Score'].sum():.2f}."
    ),
    'metadata': {'type': 'gap_summary', 'institute': 'msrit'}
})

print(f'Built {len(documents)} document chunks')
print(f'  Skill cards        : {sum(1 for d in documents if d["metadata"]["type"]=="skill_card")}')
print(f'  Trajectory summaries: {sum(1 for d in documents if d["metadata"]["type"]=="trajectory_summary")}')
print(f'  Watch List chunks  : {sum(1 for d in documents if d["metadata"]["type"]=="watch_list")}')
print(f'  Syllabus cards     : {sum(1 for d in documents if d["metadata"]["type"]=="syllabus_card")}')
print(f'  Gap summaries      : {sum(1 for d in documents if d["metadata"]["type"]=="gap_summary")}')

Built 163 document chunks
  Skill cards        : 95
  Trajectory summaries: 3
  Watch List chunks  : 0
  Syllabus cards     : 64
  Gap summaries      : 1


In [4]:
print('Loading embedding model (local, no API)...')
embedder = SentenceTransformer(EMBED_MODEL)
print('Model loaded.')

texts = [d['text'] for d in documents]
print(f'Embedding {len(texts)} chunks...')
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)
print(f'Embeddings shape: {embeddings.shape}')

Loading embedding model (local, no API)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 723.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Embedding 163 chunks...


Batches: 100%|██████████| 6/6 [00:05<00:00,  1.15it/s]

Embeddings shape: (163, 384)


In [5]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client.delete_collection(COLLECTION_NAME)
    print('Cleared existing collection.')
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)

collection.add(
    ids=[d['id'] for d in documents],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[d['metadata'] for d in documents]
)

print(f'Indexed {collection.count()} documents into ChromaDB')
print(f'Stored at: {CHROMA_PATH}')

Cleared existing collection.
Indexed 163 documents into ChromaDB
Stored at: ../outputs/chroma_db


In [ ]:
def retrieve(query: str, top_k: int = TOP_K, filter_type: str = None) -> list:
    """Embed query locally and retrieve top_k chunks from ChromaDB."""
    query_vec = embedder.encode([query])[0].tolist()
    where     = {'type': filter_type} if filter_type else None

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )

    return [
        {'text': doc, 'metadata': meta, 'distance': dist}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )
    ]


def build_context(query: str, user_type: str = 'staff') -> str:
    """
    Build retrieval context adapted to user type.
    staff   — curriculum level: gaps, debt, priority, syllabus
    student — personal level: what to learn, watch list, skill clusters
    """
    gap_chunks   = retrieve(query, top_k=1, filter_type='gap_summary')
    skill_chunks = retrieve(query, top_k=5, filter_type='skill_card')
    traj_chunks  = retrieve(query, top_k=1, filter_type='trajectory_summary')
    watch_chunks = retrieve(query, top_k=1, filter_type='watch_list')

    if user_type == 'staff':
        syl_chunks = retrieve(query, top_k=3, filter_type='syllabus_card')
        all_chunks = gap_chunks + traj_chunks + skill_chunks + syl_chunks
    else:
        all_chunks = watch_chunks + traj_chunks + skill_chunks

    return '\n\n'.join([f'[{i+1}] {c["text"]}' for i, c in enumerate(all_chunks)])


ctx = build_context('what skills are missing from cloud computing?')
print('Context preview (first 500 chars):')
print(ctx[:500])
print('...')

Context preview (first 500 chars):
[1] MSRIT curriculum gap summary. 73 of 95 industry skills are not taught at MSRIT. Data from 10 companies, 2021-2025. Top 10 by Priority Score: tensorflow, linux, css, ci/cd, networking, docker, power bi, next.js, kubernetes, terraform. Top 10 by Debt Score: aws ec2, vpc configuration, multithreading, scikit-learn, load balancing, kpi tracking, linux, excel, data structures, shell scripting. Skills in transition: distributed systems, kubernetes, microservices, next.js. Average CLI: 2.1 years. T
...


In [ ]:
SYSTEM_PROMPTS = {

    'staff': (
        "You are a Curriculum Intelligence Assistant for engineering institutes in Bangalore.\n"
        "You help faculty make data-driven decisions about curriculum updates based on industry skill demand data.\n\n"
        f"You have access to skill demand data from {TOTAL_COMPANIES} companies "
        "(Microsoft, Oracle, SAP Labs, HPE, IBM, Morgan Stanley, NatWest, BT Group, Thales, Zebra Technologies) "
        "over 2021-2025.\n"
        "Key metrics in the data:\n"
        "- CLI (Curriculum Lag Index): years a skill has been demanded before curriculum responds.\n"
        "- Priority Score: urgency to add a skill (0-1, higher = more urgent).\n"
        "- Debt Score: accumulated curriculum lag (higher = more overdue).\n"
        "- Trajectory: Traditional (established), Transitional (shifting), Upcoming (emerging).\n"
        "- Propagated Gap Score: indirectly at-risk skills via co-occurrence graph.\n\n"
        "Rules:\n"
        "- Answer ONLY from the retrieved context provided. Never invent numbers.\n"
        "- Always cite CLI, Priority Score, or Debt Score when recommending skills.\n"
        "- Mention co-occurring skills so faculty can plan clusters, not isolated additions.\n"
        "- Be specific and actionable. Faculty need to justify changes to committees.\n"
        "- If the context does not contain enough information, say: Data not available."
    ),

    'student': (
        "You are a Career Skills Advisor for engineering students in Bangalore.\n"
        "You help students understand what skills to learn based on real industry demand data.\n\n"
        f"You have access to skill demand data from {TOTAL_COMPANIES} major companies over 2021-2025.\n"
        "Key information available:\n"
        "- Which skills are Upcoming (learn now), Transitional (stable), Traditional (established).\n"
        "- A Watch List of skills that will become important soon.\n"
        "- Which skills are commonly demanded together (learn as a cluster).\n\n"
        "Rules:\n"
        "- Give practical, prioritised advice a student can act on today.\n"
        "- Recommend a clear learning order, not just a flat list.\n"
        "- Mention which companies demand each skill so students know the context.\n"
        "- Highlight Watch List skills as early opportunities to stand out.\n"
        "- Keep language simple and encouraging.\n"
        "- Answer ONLY from the retrieved context. If not in context, say: Data not available."
    )
}


def ask(query: str, user_type: str = 'staff', verbose: bool = False) -> str:
    """
    Full RAG pipeline:
    1. Embed query locally (sentence-transformers)
    2. Retrieve relevant chunks from ChromaDB (local)
    3. Build grounded prompt
    4. Generate response via Groq API (LLaMA 3, free)

    user_type: 'staff' or 'student'
    """
    context = build_context(query, user_type=user_type)

    if verbose:
        print('=== RETRIEVED CONTEXT ===')
        print(context)
        print('=== END CONTEXT ===\n')

    user_message = (
        f"Use the following retrieved curriculum intelligence data to answer the question.\n"
        f"Answer ONLY from this context. If something is not in the context, "
        f"say: Data not available.\n\n"
        f"--- RETRIEVED CONTEXT ---\n"
        f"{context}\n"
        f"--- END CONTEXT ---\n\n"
        f"Question: {query}"
    )

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPTS[user_type]},
            {'role': 'user',   'content': user_message}
        ],
        temperature=0.2,
        max_tokens=1024
    )

    return response.choices[0].message.content


print('RAG pipeline ready.')
print(f'Embeddings : sentence-transformers ({EMBED_MODEL}) — local')
print(f'Vector DB  : ChromaDB — local')
print(f'LLM        : {GROQ_MODEL} via Groq API — free')

RAG pipeline ready.
Embeddings : sentence-transformers (all-MiniLM-L6-v2) — local
Vector DB  : ChromaDB — local
LLM        : llama-3.1-8b-instant via Groq API — free


In [8]:
q = 'What skills should we add to our 6th semester cloud computing and DevOps elective?'
print(f'[STAFF] {q}\n')
print(ask(q, user_type='staff'))

[STAFF] What skills should we add to our 6th semester cloud computing and DevOps elective?

Based on the retrieved curriculum intelligence data, I recommend adding the following skills to the 6th semester cloud computing and DevOps elective:

1. **Cloud Security**: CLI: 0 years, Priority Score: 0.286, Debt Score: 0.000. This skill is currently taught at MSRIT but has a high Priority Score and is co-occurring with other skills like Kubernetes, multi-cloud, and terraform.
2. **Multi-Cloud Strategy**: CLI: 1 year, Priority Score: 0.300, Debt Score: 3.000. This skill is not taught at MSRIT and has a moderate Priority Score. It co-occurs with cloud cost optimization, cloud security, and disaster recovery planning.
3. **Cloud Cost Optimization**: CLI: 1 year, Priority Score: 0.300, Debt Score: 3.000. This skill is not taught at MSRIT and has a moderate Priority Score. It co-occurs with multi-cloud strategy, cloud security, and disaster recovery planning.
4. **Kubernetes**: CLI: 1.7 years (Tr

In [9]:
q = 'Which skills have the highest curriculum debt at MSRIT and why?'
print(f'[STAFF] {q}\n')
print(ask(q, user_type='staff'))

[STAFF] Which skills have the highest curriculum debt at MSRIT and why?

Based on the retrieved context, the top 5 skills with the highest curriculum debt at MSRIT are:

1. Security auditing (Debt Score: 9.000) - This skill has been demanded by 1 company, with a CLI of 3 years, and is missing from all 5 institutes.
2. Time series forecasting (Debt Score: 6.000) - This skill has been demanded by 1 company, with a CLI of 2 years, and is missing from all 5 institutes.
3. Load balancing (Debt Score: 5.29) - This skill is not in the provided list, but it is mentioned in the top 10 by Debt Score in the MSRIT curriculum gap summary.
4. VPC configuration (Debt Score: 4.29) - This skill is not in the provided list, but it is mentioned in the top 10 by Debt Score in the MSRIT curriculum gap summary.
5. KPI tracking (Debt Score: 4.29) - This skill is not in the provided list, but it is mentioned in the top 10 by Debt Score in the MSRIT curriculum gap summary.

These skills have significant curric

In [10]:
q = 'What emerging AI and ML skills are companies demanding that we do not currently teach?'
print(f'[STAFF] {q}\n')
print(ask(q, user_type='staff'))

[STAFF] What emerging AI and ML skills are companies demanding that we do not currently teach?

Based on the retrieved context, the emerging AI and ML skills that companies are demanding and we do not currently teach are:

1. **Generative AI**: Trajectory: Upcoming, Curriculum Lag Index: 1 year, Priority Score: 0.498, Debt Score: 2.000. Co-occurs with: LLM, Deep Learning, MLOps.
2. **LLM (Large Language Model)**: Trajectory: Upcoming, Curriculum Lag Index: 1 year, Priority Score: 0.400 (not explicitly mentioned, but implied as part of the top gap skills by priority), Debt Score: 2.000 (not explicitly mentioned, but implied as part of the top gap skills by priority). Co-occurs with: Generative AI, Deep Learning, LLM APIs.
3. **Vector Embeddings**: Trajectory: Upcoming, Curriculum Lag Index: 1 year, Priority Score: 0.300, Debt Score: 3.000. Co-occurs with: Generative AI, LLM, LLM APIs.

These skills are in high demand by companies and are part of the emerging AI and ML landscape.


In [11]:
q = 'I want to become an ML Engineer. What skills should I focus on and in what order?'
print(f'[STUDENT] {q}\n')
print(ask(q, user_type='student'))

[STUDENT] I want to become an ML Engineer. What skills should I focus on and in what order?

To become an ML Engineer, I recommend focusing on the following skills in this order:

1. **Python**: As a fundamental skill for ML, Python is already taught at MSRIT. Focus on mastering Python, especially with libraries like TensorFlow and scikit-learn.
2. **Deep Learning**: This is a Traditional skill, but still highly demanded by all 10 companies. It's already taught at MSRIT, so review and reinforce your understanding.
3. **Feature Engineering**: This Transitional skill is currently taught at MSRIT and demanded by 9 companies. It's a crucial aspect of ML, and mastering feature engineering will help you create better models.
4. **Machine Learning**: As a Transitional skill, machine learning is still in demand, and it's taught at MSRIT. Focus on understanding the concepts and applying them to real-world problems.
5. **TensorFlow**: As a library for deep learning, TensorFlow is a key skill for

In [12]:
q = 'What skills should I start learning now before they become mainstream requirements?'
print(f'[STUDENT] {q}\n')
print(ask(q, user_type='student'))

[STUDENT] What skills should I start learning now before they become mainstream requirements?

Based on the retrieved context, I recommend that you start learning the following skills now before they become mainstream requirements:

1. **Generative AI**: This skill is classified as Upcoming, meaning it's on the rise and will become increasingly important soon. It's currently in high demand by 9 out of 10 companies, with a frequency of 14. It's not taught at MSRIT, so you'll have a competitive edge if you learn it now. (Demanded by 9 of 10 companies)
2. **TensorFlow**: This skill is part of the Transitional trajectory group, which means it's stable but shifting in how it's applied. It's currently in high demand by 9 out of 10 companies, with a frequency of 9. It's not taught at MSRIT, so you'll have a competitive edge if you learn it now. (Demanded by 9 of 10 companies)
3. **CI/CD**: This skill is also part of the Transitional trajectory group and is currently in high demand by 9 out of

In [13]:
q = 'If I learn Kubernetes, what other skills should I learn alongside it?'
print(f'[STUDENT] {q}\n')
print(ask(q, user_type='student'))

[STUDENT] If I learn Kubernetes, what other skills should I learn alongside it?

If you learn Kubernetes, I recommend learning the following skills alongside it:

1. **Cloud Security**: This is an upcoming skill that is in high demand by all 10 companies, with a frequency of 19. It's a great skill to learn together with Kubernetes, as they often co-occur in the industry.
2. **Cloud Architecture**: Although its trend slope is negative, it's still a transitional skill that is demanded by 5 of 10 companies. Learning cloud architecture alongside Kubernetes will give you a well-rounded understanding of cloud computing.
3. **Terraform**: This is a traditional skill that is still in demand, with a frequency of 45. It's also a co-occurrence of Kubernetes, so learning it together will help you understand the entire cloud infrastructure.

These skills are all related to cloud computing and are in high demand by the industry. Learning them together will give you a strong foundation in cloud infra

In [ ]:
q = 'What are the top priority gaps for MSRIT?'
print(f'[DEBUG — STAFF] {q}\n')
print(ask(q, user_type='staff', verbose=True))

[DEBUG — STAFF] What are the top priority gaps for MSRIT?

=== RETRIEVED CONTEXT ===
[1] MSRIT curriculum gap summary. 73 of 95 industry skills are not taught at MSRIT. Data from 10 companies, 2021-2025. Top 10 by Priority Score: tensorflow, linux, css, ci/cd, networking, docker, power bi, next.js, kubernetes, terraform. Top 10 by Debt Score: aws ec2, vpc configuration, multithreading, scikit-learn, load balancing, kpi tracking, linux, excel, data structures, shell scripting. Skills in transition: distributed systems, kubernetes, microservices, next.js. Average CLI: 2.1 years. Total curriculum debt: 454.29.

[2] Trajectory group: Traditional. Established skills, may have declining industry relevance. 18 total skills, 11 not taught at MSRIT. Average CLI: 1.7 years. Average Priority Score: 0.411. Top gap skills by priority: terraform, microservices, multithreading, sql window functions, kpi tracking.

[3] Skill: event-driven architecture. Trajectory: Transitional. Status: NOT taught at M